In [1]:
%load_ext autoreload
%autoreload 2

import sys
import json
import math
import csv
from pathlib import Path
from collections import defaultdict
import random

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from IPython.display import Audio, display, HTML

You can use the cell below if you have access to the Discogs-VI-YT audio. 

Otherwise the notebook will use YouTube to display audio.

In [2]:
def yt_iframe(video_id, start=0, aspect="16/9"):
    return (f'<div style="position:relative; width:100%; aspect-ratio:{aspect};">'
            f'<iframe style="position:absolute; inset:0; width:100%; height:100%; border:0;" '
            f'src="https://www.youtube.com/embed/{video_id}?start={start}" '
            f'allowfullscreen></iframe></div>')


def display_row_yt(row):
    def _scalar(row, col):
        v = row[col]
        return v.item() if hasattr(v, "item") else v

    display(row)
    panels = []
    for i in (0, 1):
        yt_id = _scalar(row, f"youtube_id{i}")
        start = int(round(float(_scalar(row, f"hole_v{i}_start"))))
        panels.append(f'<div style="flex:1; min-width:0;">{yt_iframe(yt_id, start)}</div>')
    display(HTML(
        f'<div style="display:flex; gap:12px; margin:8px 0; width:100%;">'
        f'{"".join(panels)}</div>'
    ))

In [3]:
# current_dir = Path.cwd()
# parent_dir = current_dir.parent
# if str(parent_dir) not in sys.path:
#     sys.path.append(str(parent_dir))

# print(f"Added {parent_dir} to sys.path")
# from src.common.audio import load_audio, SAMPLE_RATE

# music_dir = Path("/projects/mtg/projects/unified-similarity/datasets/discogs-vi-yt-16kHz/audio/")

# def display_hole(row):

#     print(row["clique_id"].item())

#     yt_id0 = row['youtube_id0'].item()
#     yt_id1 = row['youtube_id1'].item()

#     audio_path0 = music_dir / yt_id0[:2] / f'{yt_id0}.wav'
#     audio_path1 = music_dir / yt_id1[:2] / f'{yt_id1}.wav'
    
#     segment_dur = row["segment_duration"].item()

#     hole_v0_t0 = row["hole_v0_start"].item()
#     hole_v0_t1 = min(hole_v0_t0+segment_dur, row["duration0"].item())
#     print(f"{hole_v0_t0}-{hole_v0_t1}")
#     hole_v1_t0 = row["hole_v1_start"].item()
#     hole_v1_t1 = min(hole_v1_t0+segment_dur, row["duration1"].item())
#     print(f"{hole_v1_t0}-{hole_v1_t1}")

#     audio0 = load_audio(
#         audio_path0, start=int(hole_v0_t0*SAMPLE_RATE), length=int((hole_v0_t1-hole_v0_t0)*SAMPLE_RATE)
#     )
#     audio1 = load_audio(
#         audio_path1, start=int(hole_v1_t0*SAMPLE_RATE), length=int((hole_v1_t1-hole_v1_t0)*SAMPLE_RATE)
#     )

#     display(Audio(audio0, rate=SAMPLE_RATE, normalize=False))
#     display(Audio(audio1, rate=SAMPLE_RATE, normalize=False))

## Load the dataframe

In [4]:
csv_path = Path(
    "/projects/mtg/projects/unified-similarity/dvi-similar-regions/Discogs-VI-SIREN/train/similar-regions.csv"
)
COLUMN_DTYPES = {
    "clique_id": "object",
    "version_id0": "object",
    "version_id1": "object",
    "youtube_id0": "object",
    "youtube_id1": "object",
    "duration0": "float32",
    "duration1": "float32",
    "segment_duration": "float32",
    "hole_v0_start": "float32",
    "hole_v1_start": "float32",
    "hole_height": "float32",
    "k": "int32",
    "basin_area": "int32",
}

use_fields = list(COLUMN_DTYPES.keys())
df = pd.read_csv(csv_path, usecols=use_fields, dtype=COLUMN_DTYPES)
print(f"{len(df):,}")
display(df.head(n=10))

96,754,487


,clique_id,version_id0,version_id1,youtube_id0,youtube_id1,duration0,duration1,segment_duration,k,hole_height,hole_v0_start,hole_v1_start,basin_area
0,C-0012736,V-0113562,V-0113620,-1Fe2OCHzcs,My1_VxhMUl4,163.5,118.5,20.0,0,1.741,11.0,9.0,107
1,C-0012736,V-0113562,V-0113620,-1Fe2OCHzcs,My1_VxhMUl4,163.5,118.5,20.0,1,1.860,19.0,9.0,11
2,C-0012736,V-0113562,V-0113620,-1Fe2OCHzcs,My1_VxhMUl4,163.5,118.5,20.0,2,1.868,43.0,9.0,4
3,C-0012736,V-0113562,V-0113620,-1Fe2OCHzcs,My1_VxhMUl4,163.5,118.5,20.0,3,1.875,41.0,9.0,15
4,C-0012736,V-0113562,V-0113620,-1Fe2OCHzcs,My1_VxhMUl4,163.5,118.5,20.0,4,1.877,33.0,13.0,4
5,C-0012736,V-0113562,V-0113620,-1Fe2OCHzcs,My1_VxhMUl4,163.5,118.5,20.0,5,1.887,19.0,16.0,3
6,C-0012736,V-0113562,V-0113620,-1Fe2OCHzcs,My1_VxhMUl4,163.5,118.5,20.0,6,1.924,39.0,16.0,3
7,C-0012736,V-0113562,V-0113620,-1Fe2OCHzcs,My1_VxhMUl4,163.5,118.5,20.0,7,1.932,41.0,6.0,5
8,C-0012736,V-0113562,V-0113620,-1Fe2OCHzcs,My1_VxhMUl4,163.5,118.5,20.0,8,1.932,45.0,12.0,3
9,C-0012736,V-0113562,V-0113620,-1Fe2OCHzcs,My1_VxhMUl4,163.5,118.5,20.0,9,1.950,19.0,4.0,4


## Sample a random row (a hole) and listen

In [5]:
row = df.sample(n=1)

display_row_yt(row)

# If you have the audio
# display_hole(row)

,clique_id,version_id0,version_id1,youtube_id0,youtube_id1,duration0,duration1,segment_duration,k,hole_height,hole_v0_start,hole_v1_start,basin_area
33129946,C-0045783,V-0421862,V-0421960,JDbhL62dvH0,8gSAVRl43vc,139.800003,166.5,20.0,20,0.713,6.0,57.0,12


## Get Particular Rows

You can filter by 
- clique_id (track clique id)
- version_id
- ...

### Choose a specific version pair

In [6]:
df_version_pair = df[(df['version_id0'] == "V-0434482") & (df['version_id1'] == 'V-0434495')]
display(df_version_pair.head())

,clique_id,version_id0,version_id1,youtube_id0,youtube_id1,duration0,duration1,segment_duration,k,hole_height,hole_v0_start,hole_v1_start,basin_area
56364609,C-0047224,V-0434482,V-0434495,OvYfRMnb6Fg,hf4CScIQ2Do,210.800003,286.299988,20.0,0,0.370,165.0,41.0,211
56364610,C-0047224,V-0434482,V-0434495,OvYfRMnb6Fg,hf4CScIQ2Do,210.800003,286.299988,20.0,1,0.440,165.0,72.0,151
56364611,C-0047224,V-0434482,V-0434495,OvYfRMnb6Fg,hf4CScIQ2Do,210.800003,286.299988,20.0,2,0.450,79.0,45.0,148
56364612,C-0047224,V-0434482,V-0434495,OvYfRMnb6Fg,hf4CScIQ2Do,210.800003,286.299988,20.0,3,0.455,165.0,255.0,150
56364613,C-0047224,V-0434482,V-0434495,OvYfRMnb6Fg,hf4CScIQ2Do,210.800003,286.299988,20.0,4,0.463,165.0,103.0,132


In [7]:
row = df_version_pair[df_version_pair['k'] == 0]

display_row_yt(row)
# display_hole(row)

,clique_id,version_id0,version_id1,youtube_id0,youtube_id1,duration0,duration1,segment_duration,k,hole_height,hole_v0_start,hole_v1_start,basin_area
56364609,C-0047224,V-0434482,V-0434495,OvYfRMnb6Fg,hf4CScIQ2Do,210.800003,286.299988,20.0,0,0.37,165.0,41.0,211


### Choose a range of hole height

In [8]:
d_min = 0.60
d_max = 0.70

df_hard = df[(df["hole_height"] >= d_min) & (df["hole_height"] < d_max)]
print(round(100 * len(df_hard) / len(df), 1))
display(df_hard.head())

6.6


,clique_id,version_id0,version_id1,youtube_id0,youtube_id1,duration0,duration1,segment_duration,k,hole_height,hole_v0_start,hole_v1_start,basin_area
64,C-0012736,V-0113562,V-0113683,-1Fe2OCHzcs,l6nzJvvIHvg,163.500000,166.699997,20.0,0,0.699,112.0,27.0,161
171,C-0012736,V-0113562,V-0113973,-1Fe2OCHzcs,V2RV6H_ImrY,163.500000,170.600006,20.0,0,0.693,112.0,27.0,164
535,C-0012736,V-0113579,V-0113592,TO-VbYkFqbc,7GBHpH5q-0Y,297.299988,212.300003,20.0,3,0.619,148.0,50.0,249
536,C-0012736,V-0113579,V-0113592,TO-VbYkFqbc,7GBHpH5q-0Y,297.299988,212.300003,20.0,4,0.626,20.0,142.0,319
537,C-0012736,V-0113579,V-0113592,TO-VbYkFqbc,7GBHpH5q-0Y,297.299988,212.300003,20.0,5,0.639,56.0,148.0,244


In [9]:
# Sample a random segment pair
row = df_hard.sample(n=1)

display_row_yt(row)
# display_hole(row)

,clique_id,version_id0,version_id1,youtube_id0,youtube_id1,duration0,duration1,segment_duration,k,hole_height,hole_v0_start,hole_v1_start,basin_area
12422953,C-0024971,V-0231391,V-0231471,m1NM96t7cYU,FE574iHaCyo,214.0,254.199997,20.0,18,0.607,75.0,176.0,17


### Combine Fields

In [10]:
df_safe = df[
    (df["basin_area"] > 3) &
#     (df["basin_area"] <= 55) &
    (df['k'] < 10) &
    (df['hole_height'] < 0.10)
]
print(f"{len(df_safe):,}")
print(f'{df_safe["clique_id"].nunique():,}')
print(round(100 * len(df_safe) / len(df), 1))
display(df_safe.head())

3,519,901
19,264
3.6


,clique_id,version_id0,version_id1,youtube_id0,youtube_id1,duration0,duration1,segment_duration,k,hole_height,hole_v0_start,hole_v1_start,basin_area
4618,C-0012736,V-0113683,V-0113973,l6nzJvvIHvg,V2RV6H_ImrY,166.699997,170.600006,20.0,0,0.017,145.0,145.0,155
4619,C-0012736,V-0113683,V-0113973,l6nzJvvIHvg,V2RV6H_ImrY,166.699997,170.600006,20.0,1,0.024,106.0,106.0,115
9599,C-0012736,V-0113899,V-0113900,88Sr0fseL88,s5Iq_LMirzg,249.000000,250.600006,20.0,0,0.006,19.0,19.0,192
9600,C-0012736,V-0113899,V-0113900,88Sr0fseL88,s5Iq_LMirzg,249.000000,250.600006,20.0,1,0.031,69.0,69.0,148
9601,C-0012736,V-0113899,V-0113900,88Sr0fseL88,s5Iq_LMirzg,249.000000,250.600006,20.0,2,0.036,93.0,93.0,149


In [11]:
row = df_safe.sample(n=1)

display_row_yt(row)
# display_hole(row)

,clique_id,version_id0,version_id1,youtube_id0,youtube_id1,duration0,duration1,segment_duration,k,hole_height,hole_v0_start,hole_v1_start,basin_area
8296673,C-0063390,V-0642067,V-0642289,ybswd_aICJg,loAbbzuvFa4,239.699997,268.799988,20.0,1,0.07,21.0,21.0,765
